In [23]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from collections import Counter
from urllib.parse import urlparse
import warnings
import pyphen
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

dic = pyphen.Pyphen(lang='es')
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')
sns.set_palette('husl')

warnings.filterwarnings('ignore')

print("Setup complete!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1663.75it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup complete!


In [24]:
import re

# Functions to calculate the Fernández-Huerta index
def count_syllables(word):
    word = word.lower()
    # Normalize critical diacritics
    replacements = {'á':'a', 'é':'e', 'í':'i', 'ó':'o', 'ú':'u', 'ü':'u'}
    for k, v in replacements.items():
        word = word.replace(k, v)
    # Using a hyphenation dictionary (pyphen) to count syllables
    return max(dic.inserted(word).count('-') + 1, 1)

def calculate_fernandez_huerta_readability(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    if not sentences:
        return {
            "FH_index": None,
            "readability_level": "No valid sentences",
            "syllables_per_word": None,
            "words_per_sentence": None
        }

    words = [w for s in sentences for w in re.findall(r'\b\w+\b', s)]
    total_words = len(words)

    if total_words == 0:
        return {
            "FH_index": None,
            "readability_level": "No words",
            "syllables_per_word": None,
            "words_per_sentence": None
        }

    total_syllables = sum(count_syllables(w) for w in words)
    syllables_per_word = total_syllables / total_words
    words_per_sentence = total_words / len(sentences)

    # Fernández-Huerta Formula
    index = 206.84 - (60 * syllables_per_word) - (102 * words_per_sentence)
    
    if index >= 90:
        classification = "Very easy"
    elif 80 <= index < 90:
        classification = "Easy"
    elif 70 <= index < 80:
        classification = "Somewhat easy"
    elif 60 <= index < 70:
        classification = "Normal (for adults)"
    elif 50 <= index < 60:
        classification = "Somewhat difficult"
    elif 30 <= index < 50:
        classification = "Difficult"
    else:
        classification = "Very difficult"

    return {
        "FH_index": index,
        "readability_level": classification,
        "syllables_per_word": syllables_per_word,
        "words_per_sentence": words_per_sentence
    }

def calculate_similarity(text1, text2):
    try:
        # Cosine with CountVectorizer (Bag of Words)
        vectorizer = CountVectorizer()
        vectors = vectorizer.fit_transform([text1, text2]).toarray()
        count_cosine = cosine_similarity(vectors)[0][1]

        # Cosine with Embeddings
        text1_vector = model.encode(text1)
        text2_vector = model.encode(text2)
        embeddings_cosine = cosine_similarity([text1_vector], [text2_vector])[0][0]

        # Average of similarities
        average_similarity = (count_cosine + embeddings_cosine) / 2

        return average_similarity
    except Exception as e:
        print(f"Error calculating similarity: {e}")
        return 0

In [25]:
# Load the analysis-ready datasets
df = pd.read_csv('../data/part3_analysis_ready.csv', encoding='utf-8-sig')
tools_df = pd.read_csv('../data/part3_tools_long.csv', encoding='utf-8-sig')

print(f"Analysis-ready dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Tool-long dataset shape: {tools_df.shape[0]} rows × {tools_df.shape[1]} columns")
print("\nKey canonical columns:")
print([
    'row_id', 'group_id', 'document_name_clean', 'language_code',
    'tools_normalized', 'tool_family', 'has_original_text',
    'has_proposal_text', 'is_duplicate_exact', 'quality_flags'
])


Analysis-ready dataset shape: 1589 rows × 36 columns
Tool-long dataset shape: 2317 rows × 7 columns

Key canonical columns:
['row_id', 'group_id', 'document_name_clean', 'language_code', 'tools_normalized', 'tool_family', 'has_original_text', 'has_proposal_text', 'is_duplicate_exact', 'quality_flags']


In [26]:
# Display analysis-ready dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1589 entries, 0 to 1588
Data columns (total 36 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   row_id                             1589 non-null   str    
 1   group_id                           1589 non-null   int64  
 2   document_name_raw                  1589 non-null   str    
 3   document_name_clean                1589 non-null   str    
 4   language_raw                       1468 non-null   str    
 5   language_code                      1589 non-null   str    
 6   language_label                     1589 non-null   str    
 7   tool_raw                           1586 non-null   str    
 8   tools_normalized                   1585 non-null   str    
 9   tool_primary                       1585 non-null   str    
 10  tool_secondary                     728 non-null    str    
 11  tool_count                         1589 non-null   int64  
 12  too

In [27]:
# Show a sample of normalized columns
df[[
    'row_id', 'group_id', 'document_name_clean', 'language_label',
    'tools_normalized', 'tool_family', 'has_original_text',
    'has_proposal_text', 'is_duplicate_exact', 'quality_flags'
]].head(3)


,row_id,group_id,document_name_clean,language_label,tools_normalized,tool_family,has_original_text,has_proposal_text,is_duplicate_exact,quality_flags
0,part3-bb36f63d4e2827d0,1,2020 Guide for International Students UPM,English,Gemini | ChatGPT,llm,True,True,False,multi_tool
1,part3-4d7fae99824ca7c6,1,2013 Self Assessment,English,Gemini | ChatGPT,llm,True,True,False,multi_tool
2,part3-f97464de2da94726,3,University of Manchester HR Excellence in Rese...,English,Gemini,llm,True,True,False,NaN
